## **Guia Aula01**

#### **1. Switch**

Um switch é um dispositivo de rede que conecta vários equipamentos dentro de uma mesma rede local, permitindo a comunicação entre eles.

Por que usar um switch:

* trabalha na camada 2 da rede;
* oferece melhor desempenho que hubs;
* reduz colisões;
* permite segmentação por VLANs;
* é base para redes industriais e automação.

Como usar o switch do laboratório:

* conectar o switch à energia;
* conectar notebook, CLP, ESP e outros dispositivos às portas Ethernet;
* para configuração inicial, usar a porta **CONSOLE** ou a porta **MGMT**;
* a porta **CONSOLE** é usada para acesso serial;
* a porta **MGMT** é usada para gerenciamento via Ethernet.
* O ip do switch é **192.168.1.10**, usuário: **admin**, e senha: (não tem)

O acesso inicial foi feito pela **serial console**, usando o **PuTTY**.

Passos e requisitos observados no laboratório:

* conectar na porta **CONSOLE** do switch, não na USB e nem na MGMT;
* usar o PuTTY em modo serial;
* parâmetros seriais usados:
  * **9600**
  * **8 data bits**
  * **sem paridade**
  * **1 stop bit**
  * **flow control = XON/XOFF**

Erro encontrado no acesso inicial:

**Terminal do PuTTY vazio**, sem aparecer login nem boot.

Como foi resolvido:

* o acesso correto era pela **console serial RJ45**;
* a **USB do switch não serve como console**;
* o PuTTY precisava estar configurado com os parâmetros seriais corretos;
* ao final, a comunicação serial passou a funcionar e foi possível entrar no CLI do switch.

Depois do acesso inicial, foi necessário configurar o IP da interface de gerenciamento do switch.

Comandos usados no switch:

```bash
configure vlan mgmt ipaddress 192.168.1.10 255.255.255.0
configure iproute add default 192.168.1.1 vr vr-mgmt
save configuration primary
```

O que cada comando faz:

* `configure vlan mgmt ipaddress ...` define o IP da interface de gerenciamento;
* `configure iproute add default ... vr vr-mgmt` define o gateway padrão da interface de gerenciamento;
* `save configuration primary` salva tudo para não perder após reboot.

Depois de configurar o IP, foi preciso confirmar se ele realmente estava aplicado e se a porta física de gerenciamento estava ativa.

Comandos de verificação no switch:

```bash
show ipconfig mgmt
show vlan mgmt
show port mgmt
```

Erros encontrados nessa etapa:

* ao usar `show ipconfig`, a saída veio confusa e incompleta;
* a saída também mostrou `Mgmt-port on Mgmt is down`.

Como foi resolvido:

* usar `show ipconfig mgmt`, que mostrou claramente o IP `192.168.1.10/24`;
* ajustar a conexão física da porta **MGMT** até `show port mgmt` mostrar link **ativo**, em **1000 FULL**.

No computador, o IP também precisa estar configurado corretamente.

No Windows, o caminho típico é:

`Painel de Controle -> Rede e Internet -> Central de Rede e Compartilhamento -> Alterar configurações do adaptador`

Exemplo de configuração do notebook:

* IP: `192.168.1.20`
* Máscara: `255.255.255.0`

Comando útil para verificar IP no CMD:

```bash
ipconfig
```

Depois do IP e do link da porta MGMT estarem corretos, foi feito o teste a partir do notebook.

Testes feitos:

```bash
ping 192.168.1.10
ssh admin@192.168.1.10
```

Erro encontrado no Windows:

> `General failure`
> e depois o ping não respondia.

Como foi resolvido:

* o notebook precisava estar na **mesma sub-rede** do switch;
* a porta física MGMT precisava estar ativa;
* depois que isso foi ajustado, a comunicação chegou ao ponto de tentar negociar o SSH.

Resumo dos principais comandos desta etapa:

```bash
show ipconfig mgmt
show vlan mgmt
show port mgmt
configure vlan mgmt ipaddress 192.168.1.10 255.255.255.0
configure iproute add default 192.168.1.1 vr vr-mgmt
save configuration primary
ipconfig
ping 192.168.1.10
```

#### **2. Sniffer e espelhamento de portas**

Um sniffer é uma ferramenta usada para capturar e analisar pacotes de rede. O exemplo mais comum em aula é o **Wireshark**.

Utilidade de um sniffer:

* diagnóstico de rede;
* monitoramento de tráfego;
* análise de segurança;
* engenharia reversa e estudo de protocolos.

Em um switch, o tráfego não é enviado automaticamente para todas as portas. Por isso, para capturar o tráfego de outros equipamentos, é necessário configurar **port mirroring**.

Na prática:

* as portas de origem continuam operando normalmente;
* a porta de captura recebe uma cópia dos quadros;
* o notebook com Wireshark deve ficar conectado na porta configurada como destino do espelhamento.

Isso explica por que, antes de capturar os dados, o switch precisa ser configurado para espelhar portas.

#### **3. Driver SEL C662**

O **SEL C662** é um adaptador USB para serial usado para acessar a console do switch.

Conceitos importantes:

* **RJ45**: conector físico usado pela porta console do switch;
* **DB9**: conector serial clássico;
* **RS-232**: padrão de comunicação serial.

Como verificar no Windows:

* abrir o **Gerenciador de Dispositivos**;
* localizar `Ports (COM & LPT)`;
* identificar algo como `SEL CP210x (COMX)`.

Esse `COMX` é a porta que deve ser usada no PuTTY.

Configuração prática do PuTTY:

* modo `Serial`;
* porta `COMX`;
* `9600 baud`;
* `8N1`;
* `Flow control: XON/XOFF`.

Configuração das chaves do adaptador:

* `DTE/DCE`: usar em `DTE`;
* `GND`: referência automática;
* `5V`: manter `OFF`.

Se o adaptador não aparecer corretamente no Windows, o driver deve ser baixado no site da **Schweitzer Engineering Laboratories (SEL)**.

#### **4. SSH2**

Depois que o acesso serial funcionou, o próximo passo foi habilitar o SSH2 no switch.

Comandos usados:

```bash
configure ssh2 key
enable ssh2
save configuration primary
```

Erro 1 encontrado:

Ao executar:

```bash
configure ssh2 key
```

o switch mostrou:

> `Continue? (Y/N)`
> e foi respondido **No**.

Como foi resolvido:

* executar novamente `configure ssh2 key`;
* responder **Y** para a chave ser realmente gerada.

Erro 2 encontrado:

Depois foi tentado:

```bash
show ssh2
```

e o switch respondeu:

> `Incomplete command`

Como foi resolvido:

O comando correto para verificar o estado do SSH foi:

```bash
show management
```

Foi por esse comando que apareceu a confirmação de que o SSH estava ativo:

* `SSH access : Enabled`
* `Key valid`
* `tcp port 22`

Quando a rede passou a funcionar, apareceu este erro no notebook:

```text
Unable to negotiate with 192.168.1.10 port 22:
no matching host key type found. They offer: ssh-rsa
```

Motivo:

O switch estava rodando uma versão antiga do ExtremeXOS e oferecendo apenas **`ssh-rsa`**.

A solução foi usar o cliente SSH em modo legado:

```bash
ssh -o HostKeyAlgorithms=+ssh-rsa admin@192.168.1.10
```

Se necessário, a forma mais completa seria:

```bash
ssh -o HostKeyAlgorithms=+ssh-rsa -o PubkeyAcceptedAlgorithms=+ssh-rsa admin@192.168.1.10
```

Para entender por que o comando de algoritmo não funcionava, foi verificada a versão do firmware.

Comando usado:

```bash
show version
```

Resultado:

* **ExtremeXOS 21.1.1.4**
* build de **2016**

Foi tentado:

```bash
configure ssh2 key algorithm rsa-sha2-256
```

e o switch respondeu:

> `Invalid input detected`

Conclusão:

* essa versão do EXOS é antiga demais e **não suporta esse comando**;
* as opções eram usar SSH legado no notebook ou atualizar o firmware do switch.

Resumo dos principais comandos de SSH:

```bash
configure ssh2 key
enable ssh2
save configuration primary
show management
show version
ssh -o HostKeyAlgorithms=+ssh-rsa admin@192.168.1.10
```

#### **5. Espelhamento de portas**

Como parte da configuração final, também foi mostrado como espelhar portas do switch usando SSH2.

Exemplo usado:

Espelhar as portas **33 e 34** para a **35**:

```bash
create mirror CAPTURA
configure mirror CAPTURA to port 35
configure mirror CAPTURA add port 33 ingress-and-egress
configure mirror CAPTURA add port 34 ingress-and-egress
enable mirror CAPTURA
show mirror
save configuration primary
```

Teste com **Wireshark**:

* conectar o notebook na porta `35`;
* abrir o Wireshark;
* selecionar a interface de rede correta;
* iniciar a captura.

Se a configuração estiver correta, devem aparecer quadros copiados das portas `33` e `34`.

Resumo dos principais comandos de espelhamento:

```bash
create mirror CAPTURA
configure mirror CAPTURA to port 35
configure mirror CAPTURA add port 33 ingress-and-egress
configure mirror CAPTURA add port 34 ingress-and-egress
enable mirror CAPTURA
show mirror
save configuration primary
```

#### **6. ESP-IDF**

O **ESP-IDF** é o framework oficial da Espressif para desenvolvimento com ESP32.

Por que usá-lo:

* oferece mais controle sobre hardware e rede;
* é mais adequado para projetos profissionais;
* entrega melhor integração com os recursos nativos do ESP32;
* permite trabalhar com menos abstração do que Arduino.

Comparação rápida:

| ESP-IDF | Arduino |
| --- | --- |
| Mais controle | Mais simples |
| Mais profissional | Mais amigável para iniciantes |
| Melhor integração com hardware | Mais abstração |
| Suporte mais completo | Suporte mais limitado |

No contexto desta aula, ele é importante porque conecta a parte de rede e captura de tráfego ao desenvolvimento embarcado com ESP32.

#### **7. Instalação do ESP-IDF**

Passo 1:

* abrir o VS Code;
* buscar por **ESP-IDF** nas extensões;
* instalar a extensão.

Passo 2:

* usar **Ctrl+Shift+P**;
* executar **Reload Window** depois da instalação.

Passo 3:

* abrir o comando `ESP-IDF: Configure ESP-IDF extension`;
* seguir a instalação usando **EIM** (**ESP-IDF Installation Manager**).

O que normalmente é instalado:

* Python;
* toolchain;
* ESP-IDF;
* drivers e utilitários.

Sequência prática consolidada da aula:

1. acessar o switch pela console;
2. configurar IP de gerenciamento e validar link;
3. habilitar SSH2;
4. ajustar o notebook para a mesma sub-rede;
5. testar `ping` e `ssh`;
6. configurar o espelhamento de portas;
7. capturar tráfego com Wireshark;
8. conectar isso ao contexto de sniffers e desenvolvimento em ESP-IDF.
